# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My baseline rule

My lane is **Refresh / Content Opportunity Scoring**.

The goal is to rank content items that should be reviewed first for possible refresh, improvement, protection, or monitoring.

I will use a transparent rule-based score based only on observable search and engagement signals. The score is intended for decision support, not as a guarantee that a refresh will improve performance.

My baseline prioritizes pages that have enough visibility or engagement evidence and show a clear opportunity such as staleness, declining performance, low CTR, or weak engagement.

### Scoring logic

The rule gives higher priority to pages when they have:

- meaningful search visibility;
- evidence of declining performance;
- strong visibility but low CTR;
- strong visibility but weak engagement;
- older content with meaningful search exposure.

Pages with little or no measurable evidence receive lower priority.

### Reason codes

The rule can produce these reason codes:

- `stale_visible_page` — content is old and still receives meaningful search impressions.
- `declining_with_demand` — observed performance is declining while the page still has meaningful search demand.
- `thin_visible_page` — the page has relatively low content depth but receives meaningful impressions.
- `page_one_decay_risk` — the page is ranking around page one but shows an opportunity for improvement.
- `low_ctr_visible_page` — the page receives meaningful impressions and has a relatively low CTR for its position.
- `low_engagement_visible_page` — the page receives meaningful sessions but shows weak engagement.

The baseline uses the strongest applicable reason code so that every ranked recommendation has a human-readable explanation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

#### First code cell — load March

#### Authentication/imports

In [2]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

### Load Only March

In [3]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march_df = pd.read_parquet(march_file)

print("March rows:", len(march_df))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

march_df.head()

March rows: 9841378
Date range: 2026-03-01 to 2026-03-31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Code Cell — prepare the baseline data

In [4]:
df = march_df.copy()

print("Rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Rows: 9841378
Date range: 2026-03-01 to 2026-03-31


#### Prepare a smaller March working table

In [5]:
import numpy as np
import pandas as pd
import gc

# Keep only columns needed for the baseline
cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

work = df[cols].copy()

# Release the larger dataframe
del df
gc.collect()

print("Working rows:", len(work))

Working rows: 9841378


### Aggregate March to content level

In [6]:
# Keep rows where search data is available
work = work[work["gsc_data_available"].eq(True)].copy()

# Aggregate March observations to client + content level
queue = (
    work.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        sessions=("ga4_sessions", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum"),
        days_observed=("report_date", "nunique")
    )
)

print("Unique content rows:", len(queue))
queue.head()

Unique content rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,days_observed
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.0,0.0,1
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.0,0.0,31
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,0.0,0.0,6
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.0,0.0,30
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.0,0.0,31


### Create the signals

In [7]:
queue["ctr"] = np.where(
    queue["impressions"] > 0,
    queue["clicks"] / queue["impressions"],
    np.nan
)

queue["engagement_rate"] = np.where(
    queue["sessions"] > 0,
    queue["engaged_sessions"] / queue["sessions"],
    np.nan
)

print("Signals created.")

Signals created.


### Create ONE transparent baseline score
We will use three observable signals:

visibility/demand
CTR opportunity
engagement opportunity

In [8]:
# Normalize visibility using log scale
queue["visibility_score"] = (
    np.log1p(queue["impressions"]) /
    np.log1p(queue["impressions"]).max()
)

# CTR opportunity:
# 1 = strong opportunity, 0 = no obvious CTR weakness
queue["ctr_opportunity"] = np.where(
    (queue["impressions"] >= 500) & (queue["ctr"] < 0.005),
    1.0,
    0.0
)

# Engagement opportunity
queue["engagement_opportunity"] = np.where(
    (queue["sessions"] >= 30) &
    (queue["engagement_rate"].notna()) &
    (queue["engagement_rate"] < 0.30),
    1.0,
    0.0
)

# Position opportunity:
# pages visible around page one are more useful review candidates
queue["position_opportunity"] = np.where(
    (queue["avg_position"] > 0) &
    (queue["avg_position"] <= 20) &
    (queue["impressions"] >= 100),
    1.0,
    0.0
)

# Transparent baseline score: 0–100
queue["baseline_action_score"] = (
    100 * (
        0.40 * queue["visibility_score"] +
        0.30 * queue["ctr_opportunity"] +
        0.20 * queue["engagement_opportunity"] +
        0.10 * queue["position_opportunity"]
    )
)

print(queue["baseline_action_score"].describe())

count    176738.000000
mean         29.294020
std          24.327989
min           2.079521
25%           9.133914
50%          23.875445
75%          58.790909
max          97.227622
Name: baseline_action_score, dtype: float64


### Reason code

This gives exactly one reason code per page.

In [9]:
queue["reason_code"] = "monitor"

mask = (
    (queue["impressions"] >= 500) &
    (queue["ctr"] < 0.005)
)
queue.loc[mask, "reason_code"] = "low_ctr_visible_page"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["sessions"] >= 30) &
    (queue["engagement_rate"].notna()) &
    (queue["engagement_rate"] < 0.30)
)
queue.loc[mask, "reason_code"] = "low_engagement_visible_page"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["impressions"] >= 100) &
    (queue["avg_position"] > 0) &
    (queue["avg_position"] <= 20)
)
queue.loc[mask, "reason_code"] = "page_one_opportunity"

mask = (
    (queue["reason_code"] == "monitor") &
    (queue["impressions"] >= 500)
)
queue.loc[mask, "reason_code"] = "high_visibility_monitor"

print(queue["reason_code"].value_counts())

reason_code
monitor                        87973
low_ctr_visible_page           51767
page_one_opportunity           33098
low_engagement_visible_page     3621
high_visibility_monitor          279
Name: count, dtype: int64


### Action label

In [10]:
queue["action"] = np.select(
    [
        queue["baseline_action_score"] >= 70,
        queue["baseline_action_score"] >= 40,
        queue["baseline_action_score"] >= 20
    ],
    [
        "review_refresh",
        "review",
        "monitor"
    ],
    default="low_priority"
)

print(queue["action"].value_counts())

action
low_priority      88082
review            48509
monitor           33467
review_refresh     6680
Name: count, dtype: int64


### Confidence note

In [11]:
queue["confidence_note"] = np.select(
    [
        (queue["baseline_action_score"] >= 70) &
        (queue["impressions"] >= 500),

        queue["baseline_action_score"] >= 40
    ],
    [
        "higher evidence",
        "moderate evidence"
    ],
    default="limited evidence"
)

print(queue["confidence_note"].value_counts())

confidence_note
limited evidence     121549
moderate evidence     48509
higher evidence        6680
Name: count, dtype: int64


### Rank the queue

In [12]:
queue = queue.sort_values(
    by=["baseline_action_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Ranked rows:", len(queue))

queue.head(20)

Ranked rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,days_observed,ctr,engagement_rate,visibility_score,ctr_opportunity,engagement_opportunity,position_opportunity,baseline_action_score,reason_code,action,confidence_note,rank
0,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931,669,15.008339,891.0,103.0,31,0.002731,0.115600,0.930691,1.0,1.0,1.0,97.227622,low_ctr_visible_page,review_refresh,higher evidence,1
1,client_e547b89c05043229,content_0e03de7680314cd5,221310,720,2.675217,455.0,58.0,29,0.003253,0.127473,0.923084,1.0,1.0,1.0,96.923375,low_ctr_visible_page,review_refresh,higher evidence,2
2,client_23a62021009f63c4,content_44f34c0a90047651,212404,24,7.346909,37.0,1.0,31,0.000113,0.027027,0.920004,1.0,1.0,1.0,96.800148,low_ctr_visible_page,review_refresh,higher evidence,3
3,client_e547b89c05043229,content_8d7d99f109e19aa2,203497,289,2.563756,159.0,16.0,29,0.001420,0.100629,0.916791,1.0,1.0,1.0,96.671627,low_ctr_visible_page,review_refresh,higher evidence,4
4,client_e547b89c05043229,content_4ffe18112a5642e3,186983,586,2.331060,347.0,57.0,29,0.003134,0.164265,0.910443,1.0,1.0,1.0,96.417718,low_ctr_visible_page,review_refresh,higher evidence,5
5,client_73cda7b4e4f265ea,content_471d9cabce329a66,164885,396,4.656030,99.0,1.0,31,0.002402,0.010101,0.901010,1.0,1.0,1.0,96.040398,low_ctr_visible_page,review_refresh,higher evidence,6
6,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,151166,408,3.391428,82.0,1.0,31,0.002699,0.012195,0.894495,1.0,1.0,1.0,95.779780,low_ctr_visible_page,review_refresh,higher evidence,7
7,client_73cda7b4e4f265ea,content_e241d6415ac9e534,142304,343,3.276016,128.0,0.0,31,0.002410,0.000000,0.889963,1.0,1.0,1.0,95.598536,low_ctr_visible_page,review_refresh,higher evidence,8
8,client_73cda7b4e4f265ea,content_f43118e089ecc69a,139417,191,5.036458,58.0,0.0,31,0.001370,0.000000,0.888426,1.0,1.0,1.0,95.537046,low_ctr_visible_page,review_refresh,higher evidence,9
9,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,126836,467,5.305905,155.0,4.0,31,0.003682,0.025806,0.881333,1.0,1.0,1.0,95.253313,low_ctr_visible_page,review_refresh,higher evidence,10


### Write the required CSV

In [13]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action",
    "confidence_note",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "sessions",
    "engagement_rate",
    "days_observed"
]

queue[output_columns].to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows written:", len(queue))

Saved: work/outputs/baseline_action_score.csv
Rows written: 176738


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The top 20 observations are the highest-ranked content opportunities under the transparent baseline rule.

These are decision-support recommendations, not guarantees that a refresh will improve performance.

For each observation, I review the action, reason code, confidence level, and what could make the recommendation wrong.

In [14]:
top20 = queue.head(20).copy()

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"].eq("low_ctr_visible_page"),
        top20["reason_code"].eq("low_engagement_visible_page"),
        top20["reason_code"].eq("page_one_opportunity"),
        top20["reason_code"].eq("high_visibility_monitor")
    ],
    [
        "Position or search intent may explain the low CTR.",
        "Low engagement may reflect page type, measurement coverage, or user intent.",
        "The page may have stable performance and the position signal alone may not justify a change.",
        "High visibility does not itself prove that a refresh is needed."
    ],
    default="The available signals may be insufficient without human review."
)

top20[
    [
        "rank",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
1,2,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
2,3,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
3,4,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
4,5,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
5,6,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
6,7,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
7,8,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
8,9,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...
9,10,review_refresh,low_ctr_visible_page,higher evidence,Position or search intent may explain the low ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

Some highly ranked pages may still be weak recommendations.

A high baseline score does not prove that a page needs a refresh. Seasonality, search intent, measurement coverage, or other unobserved factors could explain the observed signals.

The baseline uses March 2026 observed signals only. It does not use a future-window outcome or FlyRank product decision flag.

In [15]:
weak_picks = queue.tail(10).copy()

weak_picks[
    [
        "rank",
        "baseline_action_score",
        "reason_code",
        "action",
        "confidence_note",
        "impressions",
        "clicks",
        "sessions"
    ]
]

,rank,baseline_action_score,reason_code,action,confidence_note,impressions,clicks,sessions
176728,176729,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176729,176730,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176730,176731,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176731,176732,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176732,176733,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176733,176734,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176734,176735,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176735,176736,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176736,176737,2.079521,monitor,low_priority,limited evidence,1,0,0.0
176737,176738,2.079521,monitor,low_priority,limited evidence,1,0,0.0


### Leakage check code

In [16]:
# Columns actually used to create the ranked queue
used_columns = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "sessions",
    "engagement_rate",
    "visibility_score",
    "ctr_opportunity",
    "engagement_opportunity",
    "position_opportunity"
]

forbidden_terms = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "future",
    "target",
    "label"
]

found_forbidden = [
    term
    for term in forbidden_terms
    if any(term in col.lower() for col in used_columns)
]

print("Feature columns used:")
print(used_columns)

print("\nPotential forbidden fields:", found_forbidden)

if not found_forbidden:
    print("\nPASS: No obvious product-decision, future-window, target, or label fields were used.")
else:
    print("\nCHECK REQUIRED: Review the fields above.")

Feature columns used:
['impressions', 'clicks', 'ctr', 'avg_position', 'sessions', 'engagement_rate', 'visibility_score', 'ctr_opportunity', 'engagement_opportunity', 'position_opportunity']

Potential forbidden fields: []

PASS: No obvious product-decision, future-window, target, or label fields were used.


## Self-check

- [x] Refresh / Content Opportunity Scoring lane
- [x] One transparent baseline scoring rule
- [x] One reason code per ranked observation
- [x] One action label per ranked observation
- [x] Confidence note included
- [x] Ranked queue created
- [x] Required CSV written to work/outputs/baseline_action_score.csv
- [x] Top 20 reviewed
- [x] "What would make it wrong" included for each top-20 observation
- [x] Weak picks reviewed
- [x] Leakage check performed
- [x] No future-window outcome used
- [x] No FlyRank product decision flag used as a feature
- [x] No client names, URLs, or private queries included
- [x] Claims are decision-support claims, not guarantees